# SNOW Practitioner's Guide

**How to plan, build, validate, and troubleshoot a nonlinear photonics simulation.**

This guide is for a physicist with a chi(2) nonlinear optics problem who wants to
simulate it in SNOW.  It goes beyond the API tutorials (5–8) to teach *methodology*:
how to size your grid, estimate what regime you're in, calibrate coupling coefficients,
validate results, and diagnose failures.

**What SNOW models:**  The Nonlinear Envelope Equation (NEE) for broadband chi(2)
pulse propagation in 1D.  A single complex envelope carries all spectral components
(pump, signal, idler, harmonics).  All three-wave mixing processes — SHG, DFG, OPA,
OPO — emerge from one equation.  You set up the input conditions and the NEE handles
the physics.

**What it does not model:**  chi(3) (Kerr, Raman, SPM), diffraction, spatial modes,
material absorption beyond a bulk loss coefficient.  The simulation is 1D along the
propagation axis.

**Reference:**  M. Conforti, F. Baronio, and C. De Angelis, *Phys. Rev. A* **81**, 053841 (2010).

---

**Contents:**

1. [Planning your simulation](#part1) — grid sizing, regime estimation, backend choice
2. [Waveguide SHG recipe](#part2) — TFLN platform, with starting template
3. [Bulk crystal OPA recipe](#part3) — PPLN, X0 calibration, with starting template
4. [OPO and cavity simulations](#part4) — roundtrip loop, threshold, with starting template
5. [Validation and troubleshooting](#part5) — cross-checks, failure modes, diagnostics

---
# Part 1: Planning Your Simulation

Before writing any code, answer four questions:

1. **What wavelengths interact?**  →  Sets the bandwidth and grid.
2. **What temporal features matter?**  →  Sets N (number of grid points).
3. **What regime am I in?**  →  Estimate g·L analytically.
4. **Do I need GPU?**  →  Backend choice.

## Decision tree: "I want to simulate X"

| I want to simulate... | Recipe | Template |
|---|---|---|
| SHG in a TFLN waveguide | Waveguide + `propagate_NEE` | [Part 2](#part2) |
| OPA in a bulk crystal (PPLN, BBO, KTP) | Direct `nlo_jax.NEE` with Sellmeier dispersion | [Part 3](#part3) |
| Synchronously-pumped OPO | Roundtrip loop with feedback | [Part 4](#part4) |
| DFG / frequency conversion | Same as OPA (pump + signal in, idler out) | [Part 3](#part3) |
| Parametric fluorescence | Single-pass with `Qnoise=True`, no seed | Tutorial 7, step 2 |
| Dispersion engineering | Sweep waveguide geometry, plot GVM/GVD | Tutorial 4 |
| Temperature tuning | Recompute Sellmeier at each T, sweep | Tutorial 8, step 5 |

## Grid sizing worksheet

The simulation grid is defined by **bandwidth** (wavelength range) and **N** (number of points).
Everything else follows:

$$\text{BW} = \frac{c}{\lambda_{\text{start}}} - \frac{c}{\lambda_{\text{stop}}}, \qquad
\Delta t = \frac{1}{\text{BW}}, \qquad
T_{\text{window}} = \frac{N}{\text{BW}}$$

### Step 1: Choose the wavelength range

Include **all interacting waves** plus margin.  The simulation grid must span
every frequency that participates in the nonlinear mixing.

| Scenario | Wavelength range | Why |
|---|---|---|
| SHG: 2 µm → 1 µm | 800 nm – 3 µm | Margin below SH, above pump |
| OPA: 1.06 µm pump, 1.5 µm signal, 3.6 µm idler | 900 nm – 4.2 µm | Must reach idler |
| Degenerate OPO: 1 µm pump, ~2 µm signal | 800 nm – 3 µm | Same as SHG |

### Step 2: Choose N

N must be large enough that the **time window** resolves your temporal features
*and* accommodates group-velocity walkoff:

$$T_{\text{window}} = \frac{N}{\text{BW}} > \tau_{\text{pulse}} + \text{GVM} \times L$$

where GVM is the group-velocity mismatch between the fastest and slowest waves.

| Scenario | Typical N | Notes |
|---|---|---|
| 100 fs pulses, 4 mm TFLN, 800–3000 nm | 2$^{10}$ (1024) | T = 3.8 ps, walkoff ~1 ps |
| 100 fs pulses, 10 mm TFLN | 2$^{11}$ (2048) | Longer walkoff needs wider window |
| 10 ps pulses, 20 mm PPLN, 900–4200 nm | 2$^{14}$ (16384) | Long pulses need fine $\Delta f$ |
| OPO, many roundtrips | 2$^{10}$–2$^{12}$ | Same as single-pass SHG |

**Rule of thumb:**  If the output spectrum shows features at the grid edges,
your bandwidth is too narrow.  If temporal features wrap around the time window,
N is too small.

### Grid sizing calculator

Plug in your wavelengths and pulse duration to check whether N is sufficient.

In [ ]:
import numpy as np
from scipy.constants import c

# === EDIT THESE ===
lam_start = 800e-9     # shortest wavelength (m)
lam_stop  = 3e-6       # longest wavelength (m)
N         = 2**10      # grid points
tau_pulse = 100e-15    # pulse FWHM (s)
GVM       = 260e-15    # group-velocity mismatch (s/m) -- e.g. 260 fs/mm = 260e-12 s/m ... see below
L_crystal = 4e-3       # crystal length (m)
# ==================

BW = c/lam_start - c/lam_stop
dt = 1/BW
T_window = N / BW
df = BW / N
walkoff = abs(GVM) * L_crystal

print(f'Bandwidth:    {BW*1e-12:.1f} THz')
print(f'dt:           {dt*1e15:.2f} fs')
print(f'T_window:     {T_window*1e12:.2f} ps')
print(f'df:           {df*1e-9:.2f} GHz')
print(f'GVM walkoff:  {walkoff*1e15:.0f} fs over {L_crystal*1e3:.0f} mm')
print(f'Pulse + walkoff: {(tau_pulse + walkoff)*1e12:.2f} ps')
print()
if T_window > tau_pulse + walkoff:
    margin = T_window / (tau_pulse + walkoff)
    print(f'OK: time window is {margin:.1f}x larger than pulse + walkoff')
else:
    print(f'WARNING: time window ({T_window*1e12:.2f} ps) < pulse + walkoff ({(tau_pulse+walkoff)*1e12:.2f} ps)')
    N_needed = int(2**np.ceil(np.log2((tau_pulse + walkoff) * BW * 2)))
    print(f'  Try N = {N_needed} (2^{int(np.log2(N_needed))})')

## Pre-flight check: estimate g·L

Before running the simulation, estimate the parametric gain parameter analytically.
This tells you what regime you're in and whether the result will make physical sense.

The small-signal parametric gain coefficient for a phase-matched OPA:

$$g = \sqrt{\frac{\omega_s \, \omega_i \, d_{\text{eff}}^2 \, I_{\text{pump}}}{n_s \, n_i \, \varepsilon_0 \, c^3}}$$

The gain is $G = \cosh^2(g \cdot L)$ in the undepleted-pump, phase-matched limit.

| g·L | Regime | What to expect |
|---|---|---|
| < 1 | Weak gain | Linear amplification, seed dominates output |
| 1–3 | Clean OPA | cosh²(gL) gain, some pump depletion, seed matters |
| 3–5 | Saturation | Strong pump depletion, back-conversion begins |
| > 5 | Parametric generator | Pump spectral tail self-seeds, external seed irrelevant |

**For SHG**, the analogous figure of merit is the conversion efficiency
$\eta \propto P_{\text{pump}} \cdot L^2$ in the undepleted limit.

In [ ]:
from scipy.constants import pi, c, epsilon_0

# === EDIT THESE ===
lam_pump   = 1.064e-6     # pump wavelength (m)
lam_signal = 1.507e-6     # signal wavelength (m)
d_eff      = 17.2e-12     # effective nonlinear coefficient (m/V)
n_pump     = 2.19         # refractive index at pump
n_signal   = 2.17         # refractive index at signal
n_idler    = 2.11         # refractive index at idler
E_pump     = 5e-9         # pump pulse energy (J)
tau_pump   = 10e-12       # pump FWHM (s)
w0         = 100e-6       # beam waist radius (m) -- for bulk crystals
L          = 20e-3        # crystal length (m)
# ==================

lam_idler = 1/(1/lam_pump - 1/lam_signal)
omega_s = 2*pi*c / lam_signal
omega_i = 2*pi*c / lam_idler
P_peak = 0.88 * E_pump / tau_pump  # sech pulse
A_eff = pi * w0**2
I_pump = P_peak / A_eff

g = np.sqrt(omega_s * omega_i * d_eff**2 * I_pump / (n_signal * n_idler * epsilon_0 * c**3))
gL = g * L
G_dB = 10*np.log10(np.cosh(gL)**2)

print(f'Idler wavelength: {lam_idler*1e6:.3f} um')
print(f'Peak pump power:  {P_peak:.1f} W')
print(f'Peak intensity:   {I_pump*1e-4/1e9:.3f} GW/cm\u00b2')
print(f'g = {g:.1f} /m')
print(f'g\u00b7L = {gL:.2f}')
print(f'Analytical gain: {G_dB:.1f} dB  (cosh\u00b2, valid for g\u00b7L < 3)')
print()
if gL < 1:
    print('Regime: WEAK GAIN -- output dominated by seed')
elif gL < 3:
    print('Regime: CLEAN OPA -- cosh\u00b2 gain, moderate pump depletion')
elif gL < 5:
    print('Regime: SATURATION -- pump depletion, back-conversion likely')
else:
    print('Regime: PARAMETRIC GENERATOR -- pump spectral tail self-seeds')
    print('  External seed becomes irrelevant. Reduce pump or crystal length')
    print('  if you want to study seeded OPA.')

## Choosing the reference frame velocity

The NEE propagates in a co-moving frame at velocity `v_ref`.  The wave whose
group velocity matches `v_ref` stays centered at t=0; other waves walk off.

Choose `v_ref` to track the wave you care about most:

| Simulation | Good `v_ref` choice | Why |
|---|---|---|
| SHG | `1/wg.beta1(lam_SH)` | Watch the generated SH pulse |
| OPA (signal focused) | `1/wg.beta1(lam_signal)` | Track the amplified signal |
| OPA (idler focused) | `c / ng_idler` | Track the generated idler |
| OPO | `1/wg.beta1(lam_signal)` | Signal recirculates, must stay centered |

If you choose wrong, the wave of interest walks to the edge of the time window
and wraps around.  The fix is to increase N (wider window) or pick a better `v_ref`.

## Backend choice: CPU vs GPU

| | `backend='scipy'` (CPU) | `backend='jax'` (GPU) |
|---|---|---|
| Best for | N ≤ 2$^{12}$, single-pass, quick checks | N ≥ 2$^{12}$, sweeps, OPO roundtrips |
| First call | instant | 3–6 s compilation |
| N=2$^{14}$ single pass | ~15 min | ~5 s |
| Deterministic? | Yes | Yes (given same step sequence) |
| Agreement | bit-identical for short crystals; may diverge at high gain due to floating-point order |

**Recommendation:**  Start development with `backend='scipy'` at small N for
fast iteration.  Switch to `'jax'` for production runs at full N, parameter
sweeps, or OPO simulations.

**JAX-native poling** (`poling_fn_jax=...`) gives bit-exact results matching
the CPU solver.  Without it, the solver uses a lookup table that introduces
small discretization errors accumulating over long crystals.  Always prefer
`poling_fn_jax` when the poling pattern has an analytical form.

---
# Part 2: Waveguide SHG Recipe

The most common SNOW simulation: second-harmonic generation in a
thin-film lithium niobate (TFLN) ridge waveguide.

**Physical setup:**  A pump pulse at wavelength $\lambda_p$ enters a
periodically-poled waveguide.  Quasi-phase-matching (QPM) converts
pump photons to second-harmonic at $\lambda_p/2$.

**What you need to know:**
- Waveguide geometry (width, film thickness, etch depth)
- Material (LN, LT, ...)
- Pump wavelength and pulse parameters
- Poling period (compute from phase-matching condition)
- X0 coupling coefficient (1.1e-12 for the standard TFLN geometry)

## Computing the QPM poling period

The first-order QPM condition for SHG ($\omega + \omega \to 2\omega$):

$$\frac{n_{\text{eff}}(\lambda_p)}{\lambda_p} - \frac{n_{\text{eff}}(\lambda_p/2)}{\lambda_p/2} = \frac{1}{\Lambda}$$

where $\Lambda$ is the poling period.  For OPA/DFG ($\omega_p \to \omega_s + \omega_i$):

$$\frac{n_p}{\lambda_p} - \frac{n_s}{\lambda_s} - \frac{n_i}{\lambda_i} = \frac{1}{\Lambda}$$

In [ ]:
import snow.waveguides as waveguides
import snow.materials as materials
from scipy.constants import pi, c
nm, um, mm = 1e-9, 1e-6, 1e-3

# Waveguide geometry (standard TFLN ridge)
wg = waveguides.waveguide(w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm,
                          tf_material='LN_MgO_e', box_material='SiO2',
                          clad_material='Air')

# Compute QPM poling period for SHG: 2 um -> 1 um
lam_pump = 2*um
lam_SH = lam_pump / 2
n_pump = wg.neff(lam_pump).item()
n_SH = wg.neff(lam_SH).item()
dk_QPM = 2*n_pump/lam_pump - n_SH/lam_SH
pp = 1/dk_QPM

print(f'n_eff(pump={lam_pump/um:.1f} um) = {n_pump:.4f}')
print(f'n_eff(SH={lam_SH/um:.1f} um)    = {n_SH:.4f}')
print(f'QPM poling period: {pp/um:.2f} um')

# GVM between pump and SH
gvm = (wg.beta1(lam_SH) - wg.beta1(lam_pump)).item()
print(f'GVM (pump-SH): {gvm/(1e-15/1e-3):.1f} fs/mm')

## Starting template: Waveguide SHG

Copy this cell block to a new notebook and modify the parameters in the
`=== EDIT THESE ===` section.  Everything below runs as-is.

See **Tutorial 5** for the full worked example with detailed explanations.

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftfreq
import matplotlib.pyplot as plt

import snow.pulses as pulses
import snow.waveguides as waveguides

from scipy.constants import pi, c
nm, um, mm, ps, fs = 1e-9, 1e-6, 1e-3, 1e-12, 1e-15
MHz, THz, pJ, uW = 1e6, 1e12, 1e-12, 1e-6
plt.rcParams.update({'font.size': 14})

# ======== EDIT THESE ========
BACKEND = 'jax'         # 'scipy' for CPU, 'jax' for GPU

# Wavelength range (must include pump AND second harmonic)
lam_start = 800*nm
lam_stop  = 3*um

# Grid
N = 2**10

# Pump pulse
lam_pump = 2*um
tau      = 100*fs       # pulse FWHM
Pavg     = 1*uW         # average power
frep     = 250*MHz
N_dB     = 100          # noise floor (dB below peak)

# Waveguide
width    = 1800*nm      # ridge top width
hLN      = 700*nm       # LN film thickness
hetch    = 350*nm       # etch depth
pp       = 5.18*um      # QPM poling period
L        = 4*mm         # crystal length
X0       = 1.1e-12      # nonlinear coupling coefficient
Alpha_dBcm = 0.1        # propagation loss (dB/cm)
# ===========================

# --- Grid setup ---
BW = c/lam_start - c/lam_stop
dt = 1/BW
T_window = N/BW
t = -T_window/2 + np.arange(0, T_window, step=dt)
f = fftfreq(N, dt)
f_ref = (c/lam_start + c/lam_stop) / 2
f_abs = f + f_ref

# --- Pump pulse ---
pump = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_pump,
                         Pavg=Pavg, Npwr_dB=N_dB, frep=frep)

# --- Waveguide ---
wg = waveguides.waveguide(w_top=width, h_thinfilm=hLN, h_etch=hetch,
                          tf_material='LN_MgO_e', box_material='SiO2',
                          clad_material='Air')
wg.add_poling(lambda z: np.sign(np.cos(z * 2*pi / pp)))
wg.set_nonlinear_coeffs(N=1, X0=X0)
wg.set_length(L)
alpha = np.log(10**(Alpha_dBcm * 0.1)) * 100
wg.set_loss(alpha)

# --- Reference velocity (track the SH) ---
v_ref = 1/wg.beta1(lam_pump/2)

# --- JAX poling (for GPU backend) ---
nee_kwargs = {}
if BACKEND == 'jax':
    import jax.numpy as jnp
    nee_kwargs['poling_fn_jax'] = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))

# --- Propagate ---
out, steps = wg.propagate_NEE(pump, v_ref=v_ref, backend=BACKEND, **nee_kwargs)

# --- Energy check ---
print(f'Input energy:  {pump.energy_td()/pJ:.4f} pJ')
print(f'Output energy: {out.energy_td()/pJ:.4f} pJ')
print(f'Steps: {len(steps)}')

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
pump.plot_PSD(ax=ax1, f_unit='um')
ax1.get_lines()[-1].set(color='k', linestyle='--', alpha=0.4, label='Input')
out.plot_PSD(ax=ax1, f_unit='um')
ax1.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax1.legend(); ax1.set_title('Spectrum')

pump.plot_magsq(ax=ax2, t_unit='ps')
ax2.get_lines()[-1].set(color='k', linestyle='--', alpha=0.4, label='Input')
out.plot_magsq(ax=ax2, t_unit='ps')
ax2.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax2.legend(); ax2.set_title('Temporal Intensity')
plt.show()

## Power sweep: SHG conversion efficiency

In the undepleted-pump limit, SHG efficiency scales as $\eta \propto P \cdot L^2$.
At higher powers, the pump depletes and the efficiency saturates (and eventually
back-converts).  A power sweep maps out this curve.

In [ ]:
# Assumes waveguide `wg` and grid from the template above
powers = np.array([0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]) * uW
eta_SHG = []

lam_SH = lam_pump / 2
f0_SH = c / lam_SH

for P in powers:
    p = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_pump,
                          Pavg=P, Npwr_dB=200, frep=frep)
    o, _ = wg.propagate_NEE(p, v_ref=v_ref, verbose=False,
                            backend=BACKEND, **nee_kwargs)
    sh = o.apply_filter(f0_SH, 40*THz)
    eta = sh.energy_td() / p.energy_td()
    eta_SHG.append(eta)
    print(f'  Pavg={P/uW:7.1f} uW: eta={eta:.2e} ({10*np.log10(max(eta,1e-20)):.1f} dB)')

fig, ax = plt.subplots(figsize=(8, 5), tight_layout=True)
ax.loglog(powers/uW, eta_SHG, 'bo-')
# Undepleted-pump reference: eta \propto P
P_ref = powers[0]/uW
eta_ref = eta_SHG[0]
ax.loglog(powers/uW, eta_ref * (powers/uW) / P_ref, 'r--', alpha=0.4,
          label=r'$\eta \propto P$ (undepleted)')
ax.set_xlabel('Average power (uW)'); ax.set_ylabel('SHG efficiency')
ax.set_title(f'SHG efficiency vs pump power (L={L/mm:.0f} mm)')
ax.legend(); ax.grid(True)
plt.show()

## Poling designs

SNOW supports arbitrary poling patterns through the poling function.
Three common designs:

**Uniform QPM** — constant poling period.  Narrowband, highest peak efficiency.
```python
poling = lambda z: np.sign(np.cos(z * 2*pi / pp))
```

**Chirped QPM** — period varies linearly with z.  Broader phase-matching
bandwidth at the cost of lower peak efficiency.
```python
chirp = 0.5e-6   # period change per meter of crystal
poling = lambda z: np.sign(np.cos(z * 2*pi / (pp + chirp * z)))
```

**Apodized QPM** — Gaussian-enveloped poling strength.  Suppresses
spectral sidelobes for cleaner output.
```python
poling = lambda z: (np.exp(-((z - L/2) / (L/4))**2)
                    * np.sign(np.cos(z * 2*pi / pp)))
```

For JAX, replace `np` with `jnp` and pass via `poling_fn_jax=`.

---
# Part 3: Bulk Crystal OPA Recipe

For bulk crystals (PPLN, BBO, KTP, etc.) without waveguide confinement,
you call `nlo_jax.NEE` (or `nlo_scipy.NEE`) directly with dispersion
computed from the Sellmeier model.

**Key differences from waveguide simulations:**
- Dispersion from bulk Sellmeier equations (temperature-dependent)
- X0 must be **calibrated** against the analytical gain (no pre-calibrated value)
- Beam area enters through X0 normalization (fields are power-normalized)

## The X0 calibration procedure

The coupling coefficient X0 connects SNOW's power-normalized fields to the
physical nonlinear coefficient $d_{\text{eff}}$ and beam area.  For bulk crystals,
the first-principles formula $X_0 = 4 d_{\text{eff}} / (n \cdot c)$ gives the
wrong answer by a factor of ~20.  The correct value must be calibrated:

1. **Compute** the analytical gain $g$ from $d_{\text{eff}}$, pump intensity, and beam size
2. **Run** a weak-pump simulation ($g \cdot L < 2$, minimal pump depletion)
3. **Compare** the simulated signal gain to $\cosh^2(g \cdot L)$
4. **Adjust** X0 until they match within 1–2 dB
5. **Verify** with a seed-energy sweep: gain should be constant at low seed

For PPLN ($d_{\text{eff}} = 17.2$ pm/V) with a 100 µm beam waist,
$X_0 = 1.2 \times 10^{-14}$ matches the analytical gain.

**The factor of ~20 is not understood** — it likely relates to how the NEE's
nonlinear product maps to the standard coupled-mode parametric gain coefficient.
This is an open question in the SNOW codebase.

## Starting template: Bulk crystal OPA

Non-degenerate OPA in PPLN: 1.064 µm pump + 1.507 µm seed → 3.62 µm idler.
Copy and modify.  See **Tutorial 8** for the full worked example.

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftfreq
import matplotlib.pyplot as plt
import warnings

import snow.pulses as pulses
import snow.materials as materials

from scipy.constants import pi, c, epsilon_0
nm, um, mm, ps, fs = 1e-9, 1e-6, 1e-3, 1e-12, 1e-15
MHz, THz, nJ, pJ, uW = 1e6, 1e12, 1e-9, 1e-12, 1e-6
plt.rcParams.update({'font.size': 14})

# ======== EDIT THESE ========
BACKEND = 'jax'

# Wavelengths
lam_p = 1.064*um          # pump
lam_s = 1.507*um          # signal (seed)
lam_i = 1/(1/lam_p - 1/lam_s)  # idler (energy conservation)

# Grid
lam_start = 900*nm
lam_stop  = 4.2*um
N = 2**14                 # large N for wide BW + long pulses

# Pulses
tau      = 10*ps          # pulse FWHM
E_pump   = 5*nJ           # pump energy
E_seed   = 0.1*nJ         # seed energy
frep     = 1*MHz

# Crystal
material = 'LN_MgO_e_T'  # temperature-dependent MgO:LN
T_crystal = 150           # temperature (C)
L_crystal = 20*mm
d33      = 27e-12         # m/V (LiNbO3)
d_eff    = 2*d33/pi       # first-order QPM
w0       = 100*um         # beam waist
X0       = 1.2e-14        # calibrated coupling coefficient
# ===========================

# --- Grid setup ---
BW = c/lam_start - c/lam_stop
dt = 1/BW
T_window = N/BW
t = -T_window/2 + np.arange(0, T_window, step=dt)
f = fftfreq(N, dt)
f_ref = c/lam_s
f_abs = f + f_ref
Omega = 2*pi*f
omega_abs = 2*pi*f_abs

# --- Dispersion from Sellmeier ---
wl_grid = c / f_abs
valid = (f_abs > 0) & (wl_grid > 0.4*um) & (wl_grid < 5*um)
n_grid = np.ones(N)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    n_grid[valid] = np.array([materials.refractive_index(material, w/um, T=T_crystal)
                              for w in wl_grid[valid]])
beta_grid = 2*pi*f_abs * n_grid / c

n_s = materials.refractive_index(material, lam_s/um, T=T_crystal)
n_p = materials.refractive_index(material, lam_p/um, T=T_crystal)
n_i = materials.refractive_index(material, lam_i/um, T=T_crystal)

# Group index for reference velocity
def group_index(lam_um, T):
    dl = 0.001
    n0 = materials.refractive_index(material, lam_um, T=T)
    np_ = materials.refractive_index(material, lam_um+dl, T=T)
    nm_ = materials.refractive_index(material, lam_um-dl, T=T)
    return n0 - lam_um * (np_ - nm_)/(2*dl)

ng_s = group_index(lam_s/um, T_crystal)
v_ref = c / ng_s
beta_ref = 2*pi*f_ref * n_s / c
D = beta_grid - beta_ref - Omega/v_ref

# --- QPM poling period ---
dk_QPM = n_p/lam_p - n_s/lam_s - n_i/lam_i
pp = 1/dk_QPM
print(f'QPM poling period: {pp/um:.2f} um')
print(f'Idler wavelength: {lam_i/um:.3f} um')

# --- Coupling ---
def poling(z):
    return np.sign(np.cos(z * 2*pi / pp))

def k_func(z):
    return poling(z) * X0 * omega_abs / 4

# --- Pulses ---
pump = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_p,
                         Energy=E_pump, Npwr_dB=200, frep=frep)
seed = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_s,
                         Energy=E_seed, Npwr_dB=200, frep=frep)
input_pulse = pump + seed

# --- Propagate ---
nee_kwargs = {}
if BACKEND == 'jax':
    import jax.numpy as jnp
    import snow.nlo_jax as nlo_jax
    poling_jax = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))
    nee_kwargs['poling_fn_jax'] = poling_jax
    a_out, steps = nlo_jax.NEE(t=t, x=input_pulse.a, Omega=Omega, f0=f_ref,
                               L=L_crystal, D=D, b0=float(beta_ref),
                               b1_ref=float(1/v_ref), k=k_func,
                               verbose=True, **nee_kwargs)
else:
    import snow.nlo_scipy as nlo_scipy
    a_out, steps = nlo_scipy.NEE(t=t, x=input_pulse.a, Omega=Omega, f0=f_ref,
                                 L=L_crystal, D=D, b0=float(beta_ref),
                                 b1_ref=float(1/v_ref), k=k_func, verbose=True)

out_pulse = pulses.pulse(t, a_out, c/f_ref, frep)
print(f'{len(steps)} steps')

# --- Analyze ---
sig_out = out_pulse.apply_filter(c/lam_s, 20*THz)
idl_out = out_pulse.apply_filter(c/lam_i, 20*THz)
pump_out = out_pulse.apply_filter(c/lam_p, 20*THz)

gain_dB = 10*np.log10(sig_out.energy_td() / seed.energy_td())
pump_depl = 1 - pump_out.energy_td() / pump.energy_td()

# Analytical cross-check
A_eff = pi * w0**2
P_peak = 0.88 * E_pump / tau
I_pump = P_peak / A_eff
omega_sv = 2*pi*c/lam_s
omega_iv = 2*pi*c/lam_i
g_anal = np.sqrt(omega_sv * omega_iv * d_eff**2 * I_pump / (n_s * n_i * epsilon_0 * c**3))
G_anal_dB = 10*np.log10(np.cosh(g_anal * L_crystal)**2)

print(f'\nSignal gain:     {gain_dB:.1f} dB (simulation)')
print(f'Analytical gain: {G_anal_dB:.1f} dB (cosh\u00b2, undepleted)')
print(f'g\u00b7L = {g_anal*L_crystal:.2f}')
print(f'Pump depletion:  {pump_depl:.1%}')
print(f'Idler energy:    {idl_out.energy_td()/nJ:.3f} nJ')
print(f'Energy in={input_pulse.energy_td()/nJ:.3f}, out={out_pulse.energy_td()/nJ:.3f} nJ')

# --- Plot (dichroic-filtered) ---
f_abs_grid = fftfreq(N, t[1]-t[0]) + f_ref
def bandpass(A, center_wl, bw_THz):
    mask = np.abs(f_abs_grid - c/center_wl) < bw_THz*1e12/2
    return ifft(A * mask)

A_out = fft(out_pulse.a)
A_in = fft(input_pulse.a)
t_ps = t / ps

fig, axes = plt.subplots(2, 3, figsize=(16, 9), tight_layout=True)
DYN_RANGE = 60

for col, (wl_c, label, color) in enumerate([
    (lam_p, 'Pump', 'green'), (lam_s, 'Signal', 'blue'), (lam_i, 'Idler', 'red')]):
    # Spectrum
    ax = axes[0, col]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        if col < 2:
            input_pulse.plot_PSD(ax=ax, f_unit='um')
            ax.get_lines()[-1].set(color='k', linestyle='--', alpha=0.3, label='Input')
        out_pulse.plot_PSD(ax=ax, f_unit='um')
        ax.get_lines()[-1].set(color=color, linewidth=1.5, label='Output')
    ax.set_xlim(wl_c/um * 0.85, wl_c/um * 1.2)
    ydata = ax.get_lines()[-1].get_ydata()
    ymax = np.nanmax(ydata[np.isfinite(ydata)])
    ax.set_ylim(ymax - DYN_RANGE, ymax + 5)
    ax.legend(fontsize=9); ax.set_title(f'{label} ({wl_c/um:.3f} \u00b5m)')

    # Temporal
    ax = axes[1, col]
    a_out_f = bandpass(A_out, wl_c, 20)
    a_in_f = bandpass(A_in, wl_c, 20)
    if col < 2:
        ax.plot(t_ps, np.abs(a_in_f)**2, 'k--', alpha=0.3, label='Input')
    ax.plot(t_ps, np.abs(a_out_f)**2, color=color, linewidth=1.5, label='Output')
    ax.set_xlim(-20, 20); ax.set_xlabel('Time (ps)'); ax.set_ylabel('Power (W)')
    ax.legend(fontsize=9); ax.set_title(f'{label} temporal'); ax.grid(True)

plt.show()

---
# Part 4: OPO and Cavity Simulations

An optical parametric oscillator (OPO) is an OPA inside a cavity: the signal
is fed back to the input after each pass, building up from vacuum noise until
the round-trip gain equals the total cavity loss.

**The roundtrip loop pattern:**

```
recycled = noise(t, 0)           # start from vacuum
for rt in range(n_roundtrips):
    pump = fresh pump pulse
    signal = pulse(t, recycled)  # recycled from previous RT
    input = pump + signal
    output = propagate_NEE(input, Qnoise=True)
    signal_out = filter(output, signal_band)
    recycled = signal_out.a * sqrt(feedback_fraction)
```

**Key parameters:**

| Parameter | Meaning | Typical range |
|---|---|---|
| `Qnoise=True` | Add half-photon-per-mode vacuum noise | Always True for OPO |
| Cavity loss | Parasitic loss (mirror, scattering, absorption) | 1–20% |
| Outcoupling | Fraction extracted as output | 5–30% |
| Feedback fraction | `1 - loss - outcoupling` | What returns |
| Detuning | Timing offset between pump and returning signal | 0 to ~100 fs |
| Roundtrips to steady state | Monitor signal energy convergence | 20–60 typical |

## Starting template: Waveguide OPO

Synchronously-pumped degenerate OPO in TFLN.  Pump at 1 µm, signal at ~2 µm.
Copy and modify.  See **Tutorial 7** for the full 7-step build-up.

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftfreq
import matplotlib.pyplot as plt
import time
from IPython.display import display

import snow.pulses as pulses
import snow.waveguides as waveguides

from scipy.constants import pi, c
nm, um, mm, ps, fs = 1e-9, 1e-6, 1e-3, 1e-12, 1e-15
MHz, THz, pJ, uW = 1e6, 1e12, 1e-12, 1e-6
plt.rcParams.update({'font.size': 14})

# ======== EDIT THESE ========
BACKEND = 'jax'

# Grid
lam_start = 800*nm
lam_stop  = 3*um
N = 2**10

# Pump
lam_pump  = 1*um
tau       = 100*fs
Pavg_pump = 500*uW
frep      = 250*MHz

# Waveguide (same as SHG template)
pp   = 5.18*um
L    = 4*mm
X0   = 1.1e-12

# Cavity
outcoupling = 0.10   # 10% output coupler
cavity_loss = 0.05   # 5% parasitic loss
n_roundtrips = 30

# Signal band for filtering
f0_signal = c/(2*um)
signal_bw = 40*THz
# ===========================

feedback = 1 - outcoupling - cavity_loss

# --- Grid ---
BW = c/lam_start - c/lam_stop
dt = 1/BW
T_window = N/BW
t = -T_window/2 + np.arange(0, T_window, step=dt)
f = fftfreq(N, dt)
f_ref = (c/lam_start + c/lam_stop)/2
f_abs = f + f_ref

# --- Waveguide ---
wg = waveguides.waveguide(w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm,
                          tf_material='LN_MgO_e', box_material='SiO2',
                          clad_material='Air')
wg.add_poling(lambda z: np.sign(np.cos(z*2*pi/pp)))
wg.set_nonlinear_coeffs(N=1, X0=X0)
wg.set_length(L)
wg.set_loss(0)

v_ref = 1/wg.beta1(1.5*um)  # track the signal

# --- JAX poling ---
nee_kwargs = {}
if BACKEND == 'jax':
    import jax.numpy as jnp
    nee_kwargs['poling_fn_jax'] = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))

# --- OPO loop ---
sig_energy = np.zeros(n_roundtrips)
pump_depl = np.zeros(n_roundtrips)
recycled = pulses.noise(t, 0)  # start from vacuum

# Live plot
plt.ioff()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
line_sig, = ax1.semilogy([1], [1e-10], 'b.-')
ax1.set_xlabel('Roundtrip'); ax1.set_ylabel('Signal energy (pJ)')
ax1.set_title('OPO Build-up'); ax1.grid(True); ax1.set_xlim(1, n_roundtrips)
line_depl, = ax2.plot([1], [0], 'r.-')
ax2.set_xlabel('Roundtrip'); ax2.set_ylabel('Pump depletion (%)')
ax2.set_title('Pump Depletion'); ax2.grid(True)
ax2.set_xlim(1, n_roundtrips); ax2.set_ylim(0, 100)
fig_handle = display(fig, display_id=True)
plt.ion()

t0 = time.perf_counter()
for rt in range(n_roundtrips):
    pump = pulses.sech_pulse(t, tau, f_ref=f_ref, f0=c/lam_pump,
                             Pavg=Pavg_pump, Npwr_dB=200, frep=frep)
    sig_pulse = pulses.pulse(t, recycled, c/f_ref, frep)
    inp = pump + sig_pulse
    out, _ = wg.propagate_NEE(inp, v_ref=v_ref, verbose=False,
                              Qnoise=True, backend=BACKEND, **nee_kwargs)

    sig_rt = out.apply_filter(f0_signal, signal_bw)
    pump_out = out.apply_filter(c/lam_pump, 40*THz)
    sig_energy[rt] = sig_rt.energy_td()
    pump_depl[rt] = 1 - pump_out.energy_td()/pump.energy_td()
    recycled = sig_rt.a * np.sqrt(feedback)

    # Update live plot
    rts = np.arange(1, rt+2)
    line_sig.set_data(rts, sig_energy[:rt+1]/pJ)
    ax1.relim(); ax1.autoscale_view()
    line_depl.set_data(rts, pump_depl[:rt+1]*100)
    ax2.relim(); ax2.autoscale_view()
    fig_handle.update(fig)

elapsed = time.perf_counter() - t0
ss = slice(-10, None)
print(f'\nDone: {elapsed:.1f}s ({elapsed/n_roundtrips:.2f}s per RT)')
print(f'Steady-state signal: {np.mean(sig_energy[ss])/pJ:.3f} pJ')
print(f'Steady-state pump depletion: {np.mean(pump_depl[ss]):.1%}')
print(f'Output power: {np.mean(sig_energy[ss])*outcoupling*frep/uW:.2f} uW')

## OPO practical tips

**How many roundtrips to steady state?**  Monitor the signal energy plot.
Steady state is reached when the curve flattens.  Typical: 20–40 roundtrips.
If still growing after 60+, you may be very close to threshold.

**Threshold finding:**  Sweep pump power and check whether the OPO reaches
steady state.  The threshold is where the single-pass gain first exceeds
the total cavity loss (parasitic + outcoupling).  See Tutorial 7, step 7.

**Cavity detuning:**  In a synchronously-pumped OPO, the returning signal
must overlap temporally with the next pump pulse.  A timing offset
(detuning) of even a few fs affects the OPO for 100 fs pulses.
Apply detuning with `pulses.add_t_offset(sig_pulse, dT)` before adding
to the pump.  See Tutorial 7, step 6.

**Noise realizations:**  Because the OPO starts from vacuum noise (`Qnoise=True`),
each run produces slightly different output.  To characterize the coherence
properties, run multiple realizations and compute the first-order coherence
$|g^{(1)}|$ (see `snow.pulses.coherence_g1` if available, or compute manually
from the field autocorrelation across realizations).

---
# Part 5: Validation and Troubleshooting

Every simulation should be validated before you trust the results.
SNOW provides several levels of cross-checking.

## First check: energy conservation

**Always** compare input and output pulse energies.  For lossless propagation,
they must be equal.  With loss, the output energy should be less by the
expected amount.

```python
print(f'Input:  {input_pulse.energy_td()/pJ:.4f} pJ')
print(f'Output: {out_pulse.energy_td()/pJ:.4f} pJ')
```

If the output energy *increases* beyond the input (for Qnoise=False), something
is wrong — probably a grid/bandwidth issue.

## Analytical cross-checks

Compare your simulation against known analytical limits:

**SHG (undepleted pump):**
$$\eta_{\text{SHG}} \propto P_{\text{pump}} \cdot L^2$$
Run at low power (< 1 µW), sweep L, check quadratic scaling.

**OPA (small signal, phase-matched):**
$$G = \cosh^2(g \cdot L)$$
Run at low seed energy, compare gain to the formula.
The simulation and analytical gain should agree within 1–2 dB for $g \cdot L < 2$.

**Phase-matching bandwidth:**
$$\Delta\lambda \approx \frac{0.886 \, \lambda_s^2}{|\text{GVM}| \cdot L}$$
(sinc² acceptance bandwidth from GVM).

**Seed-energy sweep:**  At fixed pump, sweep the seed energy over many
orders of magnitude.  The gain (dB) should be constant at low seed (linear
regime) and drop at high seed (saturation/pump depletion).  See Tutorial 8,
step 2.

## Common failure modes

| Symptom | Likely cause | Fix |
|---|---|---|
| **No gain** (signal passes through unchanged) | X0 too small, wrong pump wavelength, wrong poling period, or pump and seed don't overlap temporally | Check X0 calibration, verify QPM condition, check pulse timing |
| **Everything saturates** (gain independent of parameters) | g·L >> 5, pump spectral tail self-seeds | Reduce pump energy, shorten crystal, or increase beam size |
| **Wild temporal modulation** (noisy-looking time traces) | You're seeing carrier beating between spectral components | Filter each band with `apply_filter()` or direct FFT bandpass before plotting |
| **CPU takes forever** | N too large, or many-roundtrip OPO | Use `backend='jax'`, or reduce N for initial testing |
| **JAX gives different results than CPU** | Lookup table discretization, or chaotic divergence at high gain | Use `poling_fn_jax` for bit-exact poling; verify at low gain first |
| **Output spectrum hits grid edges** | Bandwidth too narrow | Extend `lam_start`/`lam_stop` to include all interacting waves |
| **Temporal wrap-around** (features at window edges) | Time window too narrow | Increase N |
| **Filtered pulse at wrong time** (jumped to window edge) | `pulse.apply_filter()` fftshift wrapping issue | Use direct FFT filtering (see below) |
| **Energy increasing** (without Qnoise) | Numerical instability or incorrect dispersion | Check D array, reduce tolerances, verify Sellmeier range |

## Direct FFT filtering (bypass fftshift issue)

`pulse.apply_filter()` can place the filtered temporal profile at the
window boundary due to an fftshift convention mismatch.  For reliable
temporal plots of filtered spectral bands, filter in the frequency
domain directly:

```python
f_abs_grid = fftfreq(N, dt) + f_ref

def bandpass(A_fft, center_freq, bandwidth_Hz):
    """Filter in frequency domain, return time-domain field."""
    mask = np.abs(f_abs_grid - center_freq) < bandwidth_Hz / 2
    return ifft(A_fft * mask)

A_out = fft(out_pulse.a)  # FFT of output field
a_signal = bandpass(A_out, c/lam_signal, 20e12)  # 20 THz bandwidth

plt.plot(t/ps, np.abs(a_signal)**2)  # correct temporal profile
```

`apply_filter()` is fine for **energy measurements** (the energy integral
is invariant under circular shifts).  It's only the temporal *shape* that
can be wrong.

## The cross-validator

`test/cross_validate.py` provides automated CPU vs GPU validation:

```bash
# Fixed 9-level ladder (linear to full nonlinear)
python test/cross_validate.py

# Random fuzz testing (run until Ctrl-C)
python test/cross_validate.py --soak

# Bounded fuzz (100 iterations)
python test/cross_validate.py --soak -n 100
```

The validator reports the **field correlation** between CPU and GPU outputs.
All standard test cases exceed 0.999.  At high pump powers (significant
pump depletion), the correlation may drop below 0.99 due to chaotic
sensitivity to floating-point order — this is inherent to adaptive ODE
solvers, not a bug.

---
# Appendices

## A. Material properties quick reference

| Material | SNOW name | $d_{\text{eff}}$ (pm/V) | Transparency (µm) | Notes |
|---|---|---|---|---|
| 5% MgO:LN (e) | `'LN_MgO_e'` | 17.2 (QPM $2d_{33}/\pi$) | 0.4–5.0 | Workhorse for PPLN |
| 5% MgO:LN (e, T-dep) | `'LN_MgO_e_T'` | 17.2 | 0.4–5.0 | Use for temperature tuning |
| 5% MgO:LN (o) | `'LN_MgO_o'` | — | 0.4–5.0 | Ordinary axis |
| Undoped LN (e) | `'LN_e'` | ~17 | 0.4–5.0 | Lower damage threshold |
| GaP | `'GaP'` | 24 (non-QPM) | 0.6–11 | No birefringence |
| SiO$_2$ | `'SiO2'` | — | 0.2–2.5 | Waveguide cladding |
| MgO:LiTaO$_3$ | `'LT_MgO_e'` | ~10 | 0.3–5.5 | |
| Air | `'Air'` | — | all | n = 1 |

## B. Unit conversion cheatsheet

| From | To | Formula |
|---|---|---|
| dB/cm | 1/m (power) | `alpha = np.log(10**(Alpha_dBcm * 0.1)) * 100` |
| Average power + frep | Pulse energy | `E = Pavg / frep` |
| Pulse energy + FWHM | Peak power (sech) | `P_peak = 0.88 * E / tau` |
| Pulse energy + FWHM | Peak power (Gaussian) | `P_peak = 0.94 * E / tau` |
| Peak power + beam waist | Peak intensity | `I = P / (pi * w0^2)` |
| Wavelength | Frequency | `f = c / lam` |
| Group index | Group velocity | `vg = c / ng` |

SNOW uses SI units throughout: meters, seconds, Watts, Joules.

## C. Parameter glossary

| Parameter | Meaning | Where set |
|---|---|---|
| `X0` | Nonlinear coupling coefficient | `wg.set_nonlinear_coeffs()` or in `k_func` |
| `N` (mode) | Mode overlap parameter (usually 1) | `wg.set_nonlinear_coeffs(N=1)` |
| `N` (grid) | Number of grid points | User-chosen (power of 2) |
| `pp` | QPM poling period (m) | From phase-matching condition |
| `Kg` | Grating wave-vector override | `propagate_NEE(Kg=...)` |
| `Qnoise` | Add vacuum fluctuation noise | `propagate_NEE(Qnoise=True)` |
| `v_ref` | Reference frame velocity (m/s) | `1/wg.beta1(lam)` |
| `D` | Dispersion operator array | Computed from beta(f) |
| `f_ref` | Reference frequency (Hz) | `(f_max + f_min)/2` or `c/lam_signal` |
| `Npwr_dB` | Noise floor (dB below pulse peak) | `gaussian_pulse()` / `sech_pulse()` |
| `rtol`, `atol` | RK45 solver tolerances | `propagate_NEE(rtol=, atol=)` |
| `poling_fn_jax` | JAX-native poling function | `propagate_NEE(poling_fn_jax=...)` |
| `poling_table` | Pre-built poling lookup table | `nlo_jax.build_poling_table()` |

## D. Key conventions

- **Field normalization:** $|a(t)|^2$ = power in Watts.
  Energy $E = \int |a(t)|^2 \, dt$ in Joules.

- **Frequency grid:** Baseband (centered at zero).  Absolute frequencies:
  `f_abs = fftfreq(N, dt) + f_ref`.

- **FFT convention:** `pulse.a` is the time-domain field (fftshift-centered,
  t=0 in the middle).  `pulse.A` is the frequency-domain field.

- **Wavelength units:** All function arguments are in **meters** unless
  the function docstring says otherwise.  `materials.refractive_index()`
  auto-detects meters vs microns vs nm.

- **QPM coupling:** The NEE coupling is
  $k(z) = \text{poling}(z) \cdot X_0 \cdot \omega_{\text{abs}} / (4 \cdot N_{\text{mode}})$.

- **Loss:** Included via the imaginary part of the dispersion operator D,
  as $-i \alpha / 2$ (field loss coefficient = half the power loss coefficient).

For the full technical reference, see `docs/snow_reference.md`.